# OPERA RTC-S1 Fire-Scar Detection — ’t Harde

A wildfire broke out on **29 April 2026** at the **Artillerie Schietkamp**,
a military artillery range near **’t Harde, Gelderland**.
Smoke closed the A28 motorway and NL-alerts were issued across
Noord-Holland, Gelderland, and Flevoland.

This notebook uses **OPERA RTC-S1** (Sentinel-1 Radiometric Terrain
Corrected) imagery to:

1. Load pre- and post-fire passes over the burn area
2. Create **RTC colour composites** (before / after)
3. Compute a **log-ratio change map** to highlight the fire scar

## 0. Imports & Area of Interest

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from datetime import date

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

from rs_tools.config import BoundingBox
from rs_tools.datasets.loader import load_dataset, LoadedItem
from rs_tools.visualization.rtc_composite import rtc_composite

# Tight crop around the Artillerie Schietkamp, 't Harde, Gelderland
bbox = BoundingBox(west=5.72, south=52.32, east=5.90, north=52.41)

# Fire date
FIRE_DATE = date(2026, 4, 29)

## 1. Load OPERA RTC-S1 data

Fetch Sentinel-1 passes from early April through mid-May 2026 to
capture a few passes before and at least one after the fire.

In [ ]:
data = load_dataset(
    "OPERA_RTC_S1",
    bbox=bbox,
    start_date="2026-04-01",
    end_date="2026-05-15",
    archive="terrascope",
    assets=["VV", "VH"],
    limit=50,
    mosaic=True,
)

print(f"Loaded {len(data)} passes:")
for d in data:
    tag = " \u2190 POST-FIRE" if d.datetime.date() >= FIRE_DATE else ""
    print(f"  {d.label}{tag}")

before = [d for d in data if d.datetime.date() < FIRE_DATE]
after  = [d for d in data if d.datetime.date() >= FIRE_DATE]
print(f"\nPre-fire: {len(before)}  |  Post-fire: {len(after)}")

## 2. RTC Colour Composites — Before vs After

Side-by-side false-colour composites (R = VV, G = VH, B = VV).
Fire scars typically appear darker in RTC imagery because burned
vegetation scatters less radar energy.

In [ ]:
pre  = before[-1] if before else None
post = after[0]   if after  else None

if pre is not None and post is not None:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    for ax, (label, item) in zip(axes, [("Pre-fire", pre), ("Post-fire", post)]):
        item.load()
        rgb = rtc_composite(item.data["VV"].values, item.data["VH"].values)
        ax.imshow(rgb, origin="upper")
        ax.set_title(f"{label}: {item.label}", fontsize=12)
        ax.set_axis_off()
        item.unload()

    fig.suptitle(
        "Artillerie Schietkamp \u2014 't Harde wildfire (29 Apr 2026)",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()
else:
    print("Need both pre- and post-fire passes.")

## 3. All Passes — Composite Overview

Quick gallery of every loaded pass.  Passes after the fire are marked.

In [ ]:
ncols = min(len(data), 4)
nrows = int(np.ceil(len(data) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = np.atleast_2d(axes)

for idx, d in enumerate(data):
    ax = axes.flat[idx]
    d.load()
    rgb = rtc_composite(d.data["VV"].values, d.data["VH"].values)
    ax.imshow(rgb, origin="upper")
    marker = " *FIRE*" if d.datetime.date() >= FIRE_DATE else ""
    ax.set_title(f"{d.label}{marker}", fontsize=9)
    ax.set_axis_off()
    d.unload()

for idx in range(len(data), nrows * ncols):
    axes.flat[idx].set_axis_off()

fig.suptitle("OPERA RTC-S1 \u2014 't Harde (all passes)", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Change Map — Log-Ratio VV / VH

Compute the log-ratio (dB difference) between the closest pre- and
post-fire passes for both VV and VH polarisations.

* **Negative dB change** (blue) → backscatter decreased (e.g. burned veg)
* **Positive dB change** (red) → backscatter increased
* **Change magnitude** combines both channels

In [ ]:
if pre is not None and post is not None:
    pre.load()
    post.load()

    vv_pre  = pre.data["VV"].values.astype(np.float32)
    vh_pre  = pre.data["VH"].values.astype(np.float32)
    vv_post = post.data["VV"].values.astype(np.float32)
    vh_post = post.data["VH"].values.astype(np.float32)

    pre.unload()
    post.unload()

    # Log-ratio change (dB difference)
    eps = 1e-10
    vv_change_db = 10 * np.log10((vv_post + eps) / (vv_pre + eps))
    vh_change_db = 10 * np.log10((vh_post + eps) / (vh_pre + eps))

    # Mask nodata
    mask = (
        np.isnan(vv_pre) | np.isnan(vv_post)
        | np.isnan(vh_pre) | np.isnan(vh_post)
    )
    vv_change_db[mask] = np.nan
    vh_change_db[mask] = np.nan

    # Combined change magnitude
    change_mag = np.sqrt(vv_change_db**2 + vh_change_db**2)
    change_mag[mask] = np.nan

    # --- Plot ---
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))

    vmin, vmax = -5, 5
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    im0 = axes[0].imshow(vv_change_db, cmap="RdBu_r", norm=norm, origin="upper")
    axes[0].set_title(f"\u0394VV (dB)\n{pre.label} \u2192 {post.label}", fontsize=11)
    axes[0].set_axis_off()
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="dB")

    im1 = axes[1].imshow(vh_change_db, cmap="RdBu_r", norm=norm, origin="upper")
    axes[1].set_title(f"\u0394VH (dB)\n{pre.label} \u2192 {post.label}", fontsize=11)
    axes[1].set_axis_off()
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label="dB")

    im2 = axes[2].imshow(change_mag, cmap="hot_r", vmin=0, vmax=5, origin="upper")
    axes[2].set_title("Change magnitude (dB)", fontsize=11)
    axes[2].set_axis_off()
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04, label="dB")

    fig.suptitle("RTC change detection \u2014 't Harde wildfire", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Need both pre- and post-fire passes for change detection.")

## 5. Change Composite Overlay

False-colour change composite: **R = ΔVV**, **G = ΔVH**, **B = 0**.
Areas where both channels changed will appear yellow/orange.
The fire scar, if visible, should stand out against the stable surroundings.

In [ ]:
if pre is not None and post is not None:
    dvv_norm = np.clip((vv_change_db + 5) / 10, 0, 1)
    dvh_norm = np.clip((vh_change_db + 5) / 10, 0, 1)
    diff_rgb = np.dstack([dvv_norm, dvh_norm, np.zeros_like(dvv_norm)])
    diff_rgb[mask] = 1.0  # white nodata

    fig, ax = plt.subplots(figsize=(10, 9))
    ax.imshow(diff_rgb, origin="upper")
    ax.set_title(
        f"Change composite (R=\u0394VV, G=\u0394VH)\n{pre.label} \u2192 {post.label}",
        fontsize=12,
    )
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
else:
    print("Need both pre- and post-fire passes.")